# ♟️ ChessStyleNet — Colab Training on Personal Games

Train a style-conditioned chess policy network conditioned on **your own chess games** using a free Colab GPU.

**Two Ways to Load Your Games:**
- **Option A (Direct File Upload)**: Upload your exported `.pgn` file (from Lichess/Chess.com) directly into Colab.
- **Option B (Lichess API)**: Download games automatically using your exact Lichess username.

### Step 1: Environment & GPU Setup
Ensure your runtime is GPU-accelerated: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
# Clone repository (or upload your chess-style-engine folder/zip)
!git clone https://github.com/your-username/chess-style-engine.git
%cd chess-style-engine

!pip install -q -r requirements.txt
!python -c "import torch; print('CUDA Available:', torch.cuda.is_available())"

### Step 2: Prepare Your Dataset

#### Option A: Use Your Uploaded PGN File (Recommended)
1. Drag and drop your `.pgn` file (e.g. `my_games.pgn` or the file you exported from Lichess) into the Colab file explorer on the left inside `data/`.
2. Run the parser below to extract `(fen, move_uci, player_id)` training positions.

In [ ]:
!python data/pgn_to_csv.py --pgn data/my_games.pgn --out data/my_games.csv --player harikrishna

#### Option B: Download Directly from Lichess API (Alternative)
If downloading via API, note that Lichess usernames are case-sensitive and must be your exact handle.

In [ ]:
import io
import csv
import os
import requests
import chess.pgn

LICHESS_USERNAME = "your_exact_lichess_username"
MAX_GAMES = 2000

url = f"https://lichess.org/api/games/user/{LICHESS_USERNAME}?max={MAX_GAMES}&moves=true"
headers = {"Accept": "application/x-chess-pgn"}
print(f"Fetching games for '{LICHESS_USERNAME}' from Lichess...")
resp = requests.get(url, headers=headers)

if resp.status_code == 404:
    print(f"User '{LICHESS_USERNAME}' not found on Lichess! Check spelling or use Option A above.")
else:
    resp.raise_for_status()
    os.makedirs("data", exist_ok=True)
    with open("data/my_games.pgn", "w") as f:
        f.write(resp.text)
    print(f"Saved Lichess PGN! Now running parser...")
    !python data/pgn_to_csv.py --pgn data/my_games.pgn --out data/my_games.csv --player {LICHESS_USERNAME}

### Step 3: Run Quick Sanity Unit Tests

In [ ]:
!python -m pytest tests/ -v

### Step 4: Train ChessStyleNet on GPU
Trains on your personal move dataset (`data/my_games.csv`). 15-20 epochs on a Colab T4 GPU typically finishes in ~2–4 minutes.

In [ ]:
import os
os.makedirs("checkpoints", exist_ok=True)

!python src/training/train.py \
    --csv data/my_games.csv \
    --epochs 15 \
    --batch_size 256 \
    --lr 1e-3 \
    --save_path checkpoints/my_style_net.pt

### Step 5: Evaluate Model Accuracy
Check Top-1 and Top-3 accuracy on your games.

In [ ]:
!python src/evaluation/evaluate.py \
    --ckpt checkpoints/my_style_net.pt \
    --csv data/my_games.csv

### Step 6: Download Checkpoint to Run Locally
Download `my_style_net.pt` to place into your local `checkpoints/` directory for your FastAPI backend and web app.

In [ ]:
from google.colab import files
files.download("checkpoints/my_style_net.pt")